# Notebook 28 — Q4 research review and integration handoff

 This reads the **locked result files only**; it never opens Q4 raw SMART data or failure labels, trains a model, adjusts a threshold, or selects a new winner. It verifies the entire candidate → score → outcome chain, summarizes the fixed primary method beside the declared comparators, and writes a handoff for the team agent and frontend. Integration is restricted to shadow display until agent schema and operational behavior are checked.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
ROOT=COMPONENT_ROOT/'results'/RUN_ID
LOCK=ROOT/'q4_blind_candidate_protocol_v1'
SCORES=ROOT/'q4_feature_only_scoring_v1'
EVAL=ROOT/'q4_blind_failure_evaluation_v1'
OUT=ROOT/'q4_research_review_and_handoff_v1'
OUT.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
def freeze_json(path,obj):
    if path.exists():assert read_json(path)==obj,f'Existing artifact changed: {path}'
    else:path.write_text(json.dumps(obj,indent=2,allow_nan=False))
def freeze_csv(path,frame):
    b=frame.to_csv(index=False).encode('utf-8')
    if path.exists():assert path.read_bytes()==b,f'Existing artifact changed: {path}'
    else:path.write_bytes(b)
lock_path=LOCK/'q4_candidate_lock.json';lock=read_json(lock_path)
protocol=lock['protocol'];lock_complete=read_json(LOCK/'q4_protocol_lock_complete.json')
score_lock_path=SCORES/'q4_score_lock.json';score_lock=read_json(score_lock_path)
score_complete=read_json(SCORES/'q4_scoring_complete.json')
score_manifest_path=SCORES/'q4_score_manifest.json';score_manifest=read_json(score_manifest_path)
eval_complete_path=EVAL/'q4_blind_evaluation_complete.json';eval_complete=read_json(eval_complete_path)
assert lock_complete['analysis_complete'] and score_complete['analysis_complete'] and eval_complete['analysis_complete']
assert sha256(lock_path)==lock_complete['candidate_lock_sha256']==score_lock['candidate_lock_sha256']==eval_complete['candidate_lock_sha256']
assert sha256(score_lock_path)==score_complete['score_lock_sha256']==eval_complete['score_lock_sha256']
assert sha256(score_manifest_path)==score_lock['score_manifest_sha256']==score_complete['score_manifest_sha256']
assert not score_lock['failure_column_read'] and not score_lock['q4_failure_outcomes_opened']
assert protocol['primary_method']=='transformer_sector_error' and not protocol['winner_selection_after_q4']
assert not eval_complete['winner_selection_after_q4']
for key,item in lock['artifacts'].items():
    assert sha256(COMPONENT_ROOT/item['path'])==item['sha256'],key
for key,filename in [('metrics_sha256','q4_blind_failure_metrics.csv'),
                     ('event_details_sha256','q4_blind_event_details.csv'),
                     ('census_sha256','q4_first_failure_census.csv'),
                     ('summary_sha256','q4_evaluation_summary.json'),
                     ('uncertainty_sha256','q4_primary_event_recall_uncertainty.csv')]:
    assert sha256(EVAL/filename)==eval_complete[key],key
summary=read_json(EVAL/'q4_evaluation_summary.json')
assert summary['score_lock_sha256']==sha256(score_lock_path)
assert summary['first_failure_drives']==eval_complete['recorded_first_failure_drives']
assert summary['scored_endpoints']==score_lock['score_rows']
assert summary['daily_quality_totals']['scored_rows']==score_lock['score_rows']
print('Frozen candidate → feature scores → blind outcomes verified.')
print('Score rows:',score_lock['score_rows'],'| first failure drives:',summary['first_failure_drives'])

Frozen candidate → feature scores → blind outcomes verified.
Score rows: 1758728 | first failure drives: 51


## Review the frozen primary method and the declared comparators

In [2]:
metrics=pd.read_csv(EVAL/'q4_blind_failure_metrics.csv')
events=pd.read_csv(EVAL/'q4_first_failure_census.csv',dtype={'serial_number':str})
details=pd.read_csv(EVAL/'q4_blind_event_details.csv',dtype={'serial_number':str})
uncertainty=pd.read_csv(EVAL/'q4_primary_event_recall_uncertainty.csv')
models=list(lock['future_test_drives_by_model'])
methods=[protocol['primary_method']]+protocol['secondary_methods']+protocol['comparators']
horizons=[protocol['primary_horizon_days']]+protocol['other_horizons_days']
expected={(m,method,h) for m in models+['pooled_descriptive'] for method in methods for h in horizons}
assert len(metrics)==len(expected)==len(set(metrics[['group','method','horizon_days']].itertuples(index=False,name=None)))
assert set(metrics[['group','method','horizon_days']].itertuples(index=False,name=None))==expected
assert len(events)==summary['first_failure_drives'] and events.serial_number.is_unique
assert len(details)==len(events)*len(methods)*len(horizons)*2
assert set(details['serial_number'])==set(events.serial_number)
primary=metrics.loc[metrics.method.eq(protocol['primary_method']) & metrics.horizon_days.eq(14)].copy()
assert len(primary)==6 and primary.recorded_first_failure_events.sum()==2*len(events)
assert int(primary.loc[primary.group.eq('pooled_descriptive'),'events_with_complete_warning_history'].iloc[0])<=len(events)
assert len(uncertainty)==6 and set(uncertainty.group)==set(models+['pooled_descriptive'])
comparison=metrics.loc[metrics.horizon_days.eq(14),[
    'group','method','positive_endpoints','negative_endpoints','average_precision',
    'positive_prevalence','recorded_first_failure_events','events_with_complete_warning_history',
    'events_incomplete_or_unscoreable','strict_event_recall',
    'false_alerts_per_1000_observed_negative_endpoints',
    'all_scored_alerts_per_1000_endpoints','status']].copy()
comparison['frozen_primary_method']=comparison.method.eq(protocol['primary_method'])
comparison['ap_over_prevalence']=comparison.average_precision/comparison.positive_prevalence.replace(0,np.nan)
freeze_csv(OUT/'q4_14day_method_review.csv',comparison)
display(primary[['group','positive_endpoints','average_precision','positive_prevalence',
                 'recorded_first_failure_events','events_with_complete_warning_history',
                 'strict_event_recall','false_alerts_per_1000_observed_negative_endpoints']])
print('Primary method is not reselected using this comparison.')
display(comparison.loc[comparison.group.eq('pooled_descriptive'),[
    'method','average_precision','positive_prevalence','events_with_complete_warning_history',
    'strict_event_recall','false_alerts_per_1000_observed_negative_endpoints']])

,group,positive_endpoints,average_precision,positive_prevalence,recorded_first_failure_events,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints
0,TOSHIBA MG07ACA14TA,161,0.020002,0.000324,14,9,0.333333,2.439024
24,ST12000NM0008,100,0.025943,0.000345,8,6,0.333333,2.867435
48,ST4000DM000,236,0.060248,0.000816,19,15,0.333333,1.509272
72,ST8000NM0055,59,0.007758,0.000267,8,2,0.500000,2.081302
96,HGST HMS5C4040BLE640,28,0.001073,0.000145,2,2,0.000000,0.615560
120,pooled_descriptive,584,0.020189,0.000392,51,34,0.323529,2.052206


Primary method is not reselected using this comparison.


,method,average_precision,positive_prevalence,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints
120,transformer_sector_error,0.020189,0.000392,34,0.323529,2.052206
123,transformer_temp_error,0.000341,0.000392,34,0.000000,6.499884
126,transformer_joint_error,0.014482,0.000392,34,0.235294,5.427094
129,isolation_forest,0.044349,0.000392,34,0.264706,3.342644
132,sector_positive_fixed,0.017909,0.000392,34,0.529412,30.406236
135,sector_delta_fixed,0.040968,0.000392,34,0.323529,0.622715
138,sector_positive_calibrated,0.017909,0.000392,34,0.205882,2.044817
141,sector_delta_calibrated,0.040968,0.000392,34,0.323529,0.489036


## Handoff: daily agent → frozen scorer → shadow display

In [3]:
contract=read_json(COMPONENT_ROOT/lock['artifacts']['stream_contract']['path'])
assert contract['schema_version']=='research_smart_daily_v1'
assert contract['readiness'].startswith('30 consecutive valid daily observations')
assert len(models)==5
pooled=primary.loc[primary.group.eq('pooled_descriptive')].iloc[0]
assert pooled.recorded_first_failure_events==len(events)
assert pooled.events_with_complete_warning_history+pooled.events_incomplete_or_unscoreable==len(events)
# Anomaly scores are research candidates; the team agent's field names, time policy,
# SMART units and missing-value rules must be checked before any live score is trusted.
handoff={'schema_version':'research_handoff_v1','source_notebook':'HDD_28_Q4_Research_Review_and_Integration_Handoff',
         'candidate_lock_sha256':sha256(lock_path),'q4_score_lock_sha256':sha256(score_lock_path),
         'q4_blind_evaluation_complete_sha256':sha256(eval_complete_path),
         'model_checkpoint_sha256':lock['artifacts']['transformer_checkpoint']['sha256'],
         'preprocessing_sha256':lock['artifacts']['transformer_preprocessing']['sha256'],
         'primary_method':protocol['primary_method'],'primary_horizon_days':14,
         'supported_models':models,'daily_agent_input':contract['input'],
         'scoring_readiness':contract['readiness'],
         'ordering':contract['ordering'],'duplicate':contract['duplicate'],
         'frozen_transformer_thresholds':lock['thresholds']['transformer'],
         'available_output_fields':contract['output'],
         'recommended_integration_mode':'shadow display only',
         'frontend_copy':{'warming':'Collecting 30 consecutive valid daily SMART records',
                          'scored':'Research anomaly candidate; not a failure probability',
                          'unsupported_model':'Model not supported by this frozen HDD detector'},
         'regime_available':False,'drift_available':False,'ssd_validated':False,
         'automatic_healing_approved':False,'live_alert_approved':False,
         'agent_schema_verified_against_live_device':False,
         'required_checks':['map live agent fields to the five frozen SMART raw features and units',
                            'confirm one end-of-day record per serial/model and calendar-day timezone',
                            'confirm missing, duplicate, late and gap behavior with the real agent',
                            'replay representative agent records against Notebook 24 frozen scorer',
                            'check latency, persistence and display semantics before any user-facing warning'],
         'q4_context':{'reserved_first_failures':len(events),
                       'complete_primary_14day_failure_histories':int(pooled.events_with_complete_warning_history),
                       'primary_14day_recalled_events':int(pooled.events_recalled),
                       'incomplete_or_unscoreable_events':int(pooled.events_incomplete_or_unscoreable)},
         'limitations':['Q3-informed method and thresholds are frozen; Q4 must not select a new winner',
                        'few complete event histories for some models',
                        'ST4000DM000 missed tested synthetic sector-onset/ramp cases despite Q4 event flags',
                        'four provisional model names still need independent HDD type verification',
                        'no validated drift-aware regime model or SSD generalization']}
freeze_json(OUT/'research_integration_handoff.json',handoff)
review=(
    "# Q4 research decision and integration handoff\n\n"
    f"Frozen primary: `{protocol['primary_method']}` at the 14-day horizon. "
    f"Q4 scored endpoints: {score_lock['score_rows']:,}. "
    f"Reserved first failures: {len(events)}; complete warning histories: "
    f"{int(pooled.events_with_complete_warning_history)}; recalled: {int(pooled.events_recalled)}.\n\n"
    f"Primary pooled average precision: {pooled.average_precision:.5f}; "
    f"false alerts per 1,000 observed negative endpoints: "
    f"{pooled.false_alerts_per_1000_observed_negative_endpoints:.3f}. "
    "See `q4_14day_method_review.csv` for every model and frozen comparator.\n\n"
    "Decision: integrate the frozen daily scorer in shadow display with the team agent. "
    "Do not show failure probabilities, enable healing, or claim drift detection. "
    "Verify the real agent schema and 30-day replay against Notebook 24 first. "
    "The Q4 findings do not change the candidate or thresholds.\n\n"
    f"Limitations: {len(events)-int(pooled.events_with_complete_warning_history)} recorded failures "
    "lack complete 14-day warning histories; per-model event counts are small. "
    "The ST4000DM000 synthetic sector weakness remains unresolved. "
    "No SSD or validated regime/drift result is established.\n")
report_path=OUT/'q4_research_decision.md'
if report_path.exists():assert report_path.read_text()==review
else:report_path.write_text(review)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'candidate_lock_sha256':sha256(lock_path),'score_lock_sha256':sha256(score_lock_path),
        'blind_evaluation_sha256':sha256(eval_complete_path),
        'comparison_sha256':sha256(OUT/'q4_14day_method_review.csv'),
        'handoff_sha256':sha256(OUT/'research_integration_handoff.json'),
        'report_sha256':sha256(report_path),'q4_archive_reopened':False,
        'winner_selection_after_q4':False,'integration_mode':'shadow display only'}
completion_path=OUT/'review_and_handoff_complete.json'
if completion_path.exists():
    old=read_json(completion_path)
    assert {k:v for k,v in old.items() if k!='created_utc'}=={k:v for k,v in record.items() if k!='created_utc'}
else:completion_path.write_text(json.dumps(record,indent=2))
print('NOTEBOOK 28 COMPLETE — Q4 reviewed; frozen scorer ready for agent mapping and shadow frontend integration.')
print('Handoff:',OUT/'research_integration_handoff.json')

NOTEBOOK 28 COMPLETE — Q4 reviewed; frozen scorer ready for agent mapping and shadow frontend integration.
Handoff: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q4_research_review_and_handoff_v1\research_integration_handoff.json
